In [1]:
!pip install chess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 18.5 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147878 sha256=bc68c8477a61fd0ed76dada9539dafe218aaa181b80dd02700933eb8c1e926e8
  Stored in directory: /root/.cache/pip/wheels/74/65/99/8c3a88ba852c1ed8d34daed144b8cb4c654d638677483d19ff
Successfully built chess


In [2]:
import chess
import math
import random

In [3]:
def getLegalMoves(board): #I don't know why i made this a function, but i don't want to overthink it so it's staying
  moves = []
  for move in board.legal_moves:
    moves.append(move.uci())
  return moves

In [4]:
# board = chess.Board() #was only used at the start for testing

# Moves = list(getLegalMoves(board))
# print(Moves)

# print(board)

In [5]:
#tables with value adjustments based on piece positions
#lower values mean a piece at that spot is less useful than at a spot with a higher value. e.g. pawn in the middle (e4,d4,e5,d5) is more useful than one at the edge
pawntable = [ #values from https://chessprogramming.org/Simplified_Evaluation_Function

    0,  0,  0,  0,  0,  0,  0,  0,
    50, 50, 50, 50, 50, 50, 50, 50,
    10, 10, 20, 30, 30, 20, 10, 10,
     5,  5, 10, 25, 25, 10,  5,  5,
     0,  0,  0, 20, 20,  0,  0,  0,
     5, -5,-10,  0,  0,-10, -5,  5,
     5, 10, 10,-20,-20, 10, 10,  5,
     0,  0,  0,  0,  0,  0,  0,  0
]

knightstable = [
    -50,-40,-30,-30,-30,-30,-40,-50,
    -40,-20,  0,  0,  0,  0,-20,-40,
    -30,  0, 10, 15, 15, 10,  0,-30,
    -30,  5, 15, 20, 20, 15,  5,-30,
    -30,  0, 15, 20, 20, 15,  0,-30,
    -30,  5, 10, 15, 15, 10,  5,-30,
    -40,-20,  0,  5,  5,  0,-20,-40,
    -50,-40,-30,-30,-30,-30,-40,-50,
]

bishopstable = [
    -20,-10,-10,-10,-10,-10,-10,-20,
    -10,  0,  0,  0,  0,  0,  0,-10,
    -10,  0,  5, 10, 10,  5,  0,-10,
    -10,  5,  5, 10, 10,  5,  5,-10,
    -10,  0, 10, 10, 10, 10,  0,-10,
    -10, 10, 10, 10, 10, 10, 10,-10,
    -10,  5,  0,  0,  0,  0,  5,-10,
    -20,-10,-10,-10,-10,-10,-10,-20,
]

rookstable = [
      0,  0,  0,  0,  0,  0,  0,  0,
      5, 10, 10, 10, 10, 10, 10,  5,
     -5,  0,  0,  0,  0,  0,  0, -5,
     -5,  0,  0,  0,  0,  0,  0, -5,
     -5,  0,  0,  0,  0,  0,  0, -5,
     -5,  0,  0,  0,  0,  0,  0, -5,
     -5,  0,  0,  0,  0,  0,  0, -5,
      0,  0,  0,  5,  5,  0,  0,  0
]

queenstable = [
    -20,-10,-10, -5, -5,-10,-10,-20,
    -10,  0,  0,  0,  0,  0,  0,-10,
    -10,  0,  5,  5,  5,  5,  0,-10,
     -5,  0,  5,  5,  5,  5,  0, -5,
      0,  0,  5,  5,  5,  5,  0, -5,
    -10,  5,  5,  5,  5,  5,  0,-10,
    -10,  0,  5,  0,  0,  0,  0,-10,
    -20,-10,-10, -5, -5,-10,-10,-20
]

kingstable = [
    -30,-40,-40,-50,-50,-40,-40,-30,
    -30,-40,-40,-50,-50,-40,-40,-30,
    -30,-40,-40,-50,-50,-40,-40,-30,
    -30,-40,-40,-50,-50,-40,-40,-30,
    -20,-30,-30,-40,-40,-30,-30,-20,
    -10,-20,-20,-20,-20,-20,-20,-10,
     20, 20,  0,  0,  0,  0, 20, 20,
     20, 30, 10,  0,  0, 10, 30, 20
]

In [6]:
pstDict = { #dict to bind the PSTs to pieces, maybe python isn't that useless. dicts are fun
    chess.PAWN: pawntable,
    chess.KNIGHT: knightstable,
    chess.BISHOP: bishopstable,
    chess.ROOK: rookstable,
    chess.QUEEN: queenstable,
    chess.KING: kingstable
}

In [7]:
def evaluateBoard(board):
  #taken from https://healeycodes.com/building-my-own-chess-engine
  piece_values = { #sets piece weights/values so the ai prioritizes attacking opponents higher value ones and preserving their own high value pieces
    chess.PAWN: 100,
    chess.ROOK: 500,
    chess.KNIGHT: 320,
    chess.BISHOP: 330,
    chess.QUEEN: 900,
    chess.KING: 20000
  }
  white = 0
  black = 0

  for square in chess.SQUARES: #checks the piece at every square on the table adds the value along with the Piece square table value so the positioning of pieces is considered during the evaluation
    piece = board.piece_at(square)
    if piece:
      table = pstDict[piece.piece_type]
      if piece.color == chess.WHITE:
        white += (piece_values[piece.piece_type] + table[square])
      else:
        black += (piece_values[piece.piece_type] + table[chess.square_mirror(square)]) #mirrors the current square vertically so the PSTs still fit.
    else:
      continue

  # print("white", white)
  # print("black", black)


  return white - black #returns difference between white and black evaluation
  #positive = white is doing better
  #negative black is doing better
  #0 = it's even

In [8]:
def minimax(board, depth, alpha, beta, max_colour):
  if(depth == 0 or board.is_game_over()): #end conditions
    return None, evaluateBoard(board) #if no moves avilable (game ended or depth is 0) send back final evaluation of the board

  moves = getLegalMoves(board)
  bestMove = random.choice(moves) #picks random move to avoid the ai picking the exact same move if evaluation is 0

  if(max_colour):
    maxEval = -math.inf
    for move in moves:
      board.push_uci(move)
      currentEval = minimax(board, depth-1, alpha, beta, False)[1] #for each legal move the program does it, calls itself (function), returns final evaluation after all next possible moves are taken? it's late idk

      # print("currentEval ", currentEval, " move ", move)

      board.pop() #takes back the move to return the board to the state it was in before the minimax function was called

      if(currentEval > maxEval): #checks if the current evaluation is better for white than the previous maxEval, if so assign maxEval to currentEval, and bestMove to current move.
        maxEval = currentEval
        bestMove = move

      #-------
      #code taken from youtube.com/watch?v=-ivz8yJ4l4E
      alpha = max(alpha, currentEval)
      if(beta <= alpha):
        break

    return bestMove, maxEval #returns best move(next possible move to play) found and evaluation(for curiosity reason)
  else:
    minEval = math.inf #does the same as above but min instead of max so change > to <
    for move in moves:
      board.push_uci(move)
      currentEval = minimax(board, depth-1, alpha, beta, True)[1]
      # print("currentEval ", currentEval, " move ", move)

      board.pop()
      if(currentEval < minEval):
        minEval = currentEval
        bestMove = move
      #-------
      #code taken from youtube.com/watch?v=-ivz8yJ4l4E
      beta = min(beta, currentEval)
      if(beta <= alpha):
        break

    return bestMove, minEval

In [9]:
colour = True
board = chess.Board() #reinitializes the board so i can only rerun this section instead of rerunning the entire program
# board.push_uci('e2e4') #gives is an intial move because it does dumb crap when board is in default position.
# board.push_uci('e7e5')
# board.push_uci('g1f3')
# board.push_uci('b8c6')
# board.push_uci('f1c4')
#possible fix is to have an array/list/touple or what have you with moves that give the same evaluation and the pick a random one from the list. kind of shit but it would fix the problem

# board.push_uci('e7e5')
# board.push_uci('g1f3')
# board.push_uci('d7d6')
# board.push_uci('d2d4')
# board

while(True):

  if(colour == True):
    print('make your move')
    move = input()
    legalMoves = getLegalMoves
    if move in legalMoves(board):
      board.push_uci(move)
      colour = False
  else:
    bestMove, eval = minimax(board, 4, -(math.inf), (math.inf), colour)
    board.push_uci(bestMove)
    print(bestMove)
    colour = True

for i in range(80): #calls minimax 10 times and moves to board
  bestMove, eval = minimax(board, 4, -(math.inf), (math.inf), colour) #only depth 3 because 4+ takes ages to run (very unoptimized program) and 2 gives much worse results.
  #now added alpha-beta pruning seemingly succesffuly. cut a bit of the time off. still takes too long on 5 depth. https://www.youtube.com/watch?v=-ivz8yJ4l4E and https://chessprogramming.org/Minimax
  # if (i%2) == 0:
  #   print('-------')
  print(bestMove)#, "eval", eval)
  board.push_uci(bestMove)
  if colour:
    colour = False
  else:
    colour = True #changes the colour after each move is made


make your move
e2e4
b8c6
make your move


KeyboardInterrupt: Interrupted by user